# Passo a Passo do pyodbc <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/microsoftsqlserver/microsoftsqlserver-plain.svg" height="45" />🧭

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Docker](https://img.shields.io/badge/Docker-111827?style=flat-square&logo=docker&logoColor=2496ED)
![pyodbc](https://img.shields.io/badge/pyodbc-0078D4?style=flat-square)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-pyodbc%20%7C%20cursor%20%7C%20crud-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-container%20docker%20de%20pé-purple)
![Biblioteca](https://img.shields.io/badge/requer-pyodbc-orange)

> Um giro completo pelo ciclo de vida de uma conexão `pyodbc`: conectar, criar uma tabela, inserir, ler, atualizar, apagar e fechar — tudo numa tabela descartável, só pra fixar o fluxo antes de aplicar em dados de verdade nos próximos notebooks.

## 📋 Conteúdo

1. [Subindo o Banco com Docker](#-1-subindo-o-banco-com-docker)
2. [Conectando](#-2-conectando)
3. [Criando uma Tabela de Teste](#-3-criando-uma-tabela-de-teste)
4. [Inserindo Dados](#-4-inserindo-dados)
5. [Lendo os Dados](#-5-lendo-os-dados)
6. [Atualizando um Registro](#-6-atualizando-um-registro)
7. [Apagando um Registro](#-7-apagando-um-registro)
8. [Fechando a Conexão](#-8-fechando-a-conexão)

## 🐳 1. Subindo o Banco com Docker

O `SQLSERVER_HOST` de `conexao.py` aponta pra `localhost,1433` — ou seja, precisa de um SQL Server rodando localmente *antes* da primeira célula. Quem sobe isso é o `docker-compose.yml` da pasta `docker/`, que usa a imagem **Azure SQL Edge** (o motor SQL Server "de nuvem" da Microsoft, leve o suficiente pra rodar em container/edge, inclusive em Mac com Apple Silicon).

```bash
# a partir da pasta docker/
docker compose up -d sqlserver

# se o container já existe mas está parado
docker start hashtag-sqlserver
```

| Comando 🔑 | O que faz 🔓 |
|---|---|
| `docker compose up -d sqlserver` | Cria (ou recria) o container `hashtag-sqlserver`, em background |
| `docker start hashtag-sqlserver` | Só liga um container que já existia e foi parado |
| `docker ps` | Confirma se o container está `Up` antes de tentar conectar |

> ⚠️ Se essa etapa for pulada, `nova_conexao_sqlserver()` estoura erro de conexão — o driver não acha ninguém ouvindo em `localhost,1433`.


## 🔌 2. Conectando

A partir daqui, a conexão vem de `conexao.py` — o módulo que centraliza host, usuário e senha (ver notebook anterior). `nova_conexao_sqlserver()` já devolve a conexão pronta, apontando pro banco `HashtagCursoSQL`.

In [1]:
from conexao import nova_conexao_sqlserver
from cores import *

conexao = nova_conexao_sqlserver(banco="HashtagCursoSQL")
cursor = conexao.cursor()

print(f"{CinzaClaro}Conexão{Reset} {Verde}aberta{Reset} {CinzaClaro}com o banco HashtagCursoSQL{Reset}")

Conexão aberta com o banco HashtagCursoSQL


## 🏗️ 2. Criando uma Tabela de Teste

`cursor.execute()` roda qualquer comando SQL — inclusive `CREATE TABLE`. O `IF OBJECT_ID(...) IS NOT NULL DROP TABLE` no início deixa a célula segura pra rodar de novo (sem erro de "tabela já existe").

| Método 🔑 | O que faz 🔓 |
|---|---|
| `cursor.execute(sql)` | Envia um comando SQL pro servidor |
| `conexao.commit()` | Confirma a mudança de estrutura/dados no banco |

In [2]:
cursor.execute("""
IF OBJECT_ID('dbo.TesteConexao', 'U') IS NOT NULL
    DROP TABLE dbo.TesteConexao;

CREATE TABLE dbo.TesteConexao (
    Id INT IDENTITY(1,1) PRIMARY KEY,
    Mensagem VARCHAR(100) NOT NULL,
    CriadoEm DATETIME NOT NULL DEFAULT GETDATE()
);
""")
conexao.commit()

print(f"{CinzaClaro}Tabela TesteConexao{Reset} {Verde}criada.{Reset}")

Tabela TesteConexao criada.


## ➕ 3. Inserindo Dados

`INSERT INTO` com `?` no lugar dos valores — o parâmetro vai depois, como argumento de `execute()`. É a forma segura (ver notebook anterior sobre SQL Injection).

In [3]:
cursor.execute(
    "INSERT INTO dbo.TesteConexao (Mensagem) VALUES (?)",
    "Primeira linha inserida via pyodbc"
)
conexao.commit()

print(f"{VerdeClaro}Linha inserida.{Reset} {CinzaClaro}Linhas afetadas:{Reset} {MagentaClaro}{cursor.rowcount}{Reset}")


Linha inserida. Linhas afetadas: 1


## 📖 4. Lendo os Dados

Depois de um `SELECT`, o resultado não vem sozinho — é preciso "puxar" as linhas do cursor com `fetchall()`, `fetchone()` ou iterando direto.

| Método 🔑 | Devolve 🔓 |
|---|---|
| `cursor.fetchall()` | todas as linhas, numa lista |
| `cursor.fetchone()` | a próxima linha só |

In [4]:
cursor.execute("SELECT Id, Mensagem, CriadoEm FROM dbo.TesteConexao")
linhas = cursor.fetchall()

for indice_linha, linha in enumerate(linhas):
    print(f"{CinzaClaro}#{linha.Id}{Reset} {VerdeClaro}{linha.Mensagem}{Reset} {CinzaClaro}({linha.CriadoEm}){Reset}")


#1 Primeira linha inserida via pyodbc (2026-08-17 21:21:43.420000)


## ✏️ 5. Atualizando um Registro

`UPDATE` segue o mesmo padrão: SQL com `?`, valores como parâmetros, e `commit()` no final pra confirmar.

In [5]:
cursor.execute(
    "UPDATE dbo.TesteConexao SET Mensagem = ? WHERE Id = ?",
    "Mensagem atualizada via pyodbc", 1
)
conexao.commit()

cursor.execute("SELECT Id, Mensagem FROM dbo.TesteConexao WHERE Id = ?", 1)
linha_atualizada = cursor.fetchone()

print(f"{VerdeClaro}Atualizado:{Reset} #{linha_atualizada.Id} {linha_atualizada.Mensagem}")


Atualizado: #1 Mensagem atualizada via pyodbc


## 🗑️ 6. Apagando um Registro

`DELETE FROM ... WHERE` — sem o `WHERE`, apaga a tabela inteira. Sempre confirme a condição antes de rodar.

In [6]:
cursor.execute("DELETE FROM dbo.TesteConexao WHERE Id = ?", 1)
conexao.commit()

cursor.execute("SELECT COUNT(*) AS Total FROM dbo.TesteConexao")
total_restante = cursor.fetchone().Total

print(f"{VermelhoClaro}Linha apagada.{Reset} {CinzaClaro}Linhas restantes na tabela:{Reset} {MagentaClaro}{total_restante}{Reset}")


Linha apagada. Linhas restantes na tabela: 0


## 🔚 7. Fechando a Conexão

Ciclo completo: conectar → criar → inserir → ler → atualizar → apagar → **fechar**.

In [7]:
conexao.close()
print(f"{CinzaClaro}Conexão fechada.{Reset}")


Conexão fechada.


Esse é o padrão que se repete em praticamente todo script `pyodbc`. Os próximos 5 notebooks pegam cada uma dessas operações (Create, Read, Read com pandas, Update, Delete) e aprofundam, já numa tabela de verdade (`Vendas`), que fica de pé pro resto do módulo.

> ▶️ Próximo notebook: **Create no Banco de Dados**.